# 2. Results

Reads what `scripts/evaluate.py` measured on the held-out test patients.
Nothing is computed here that is not also reproducible from the command line -
this notebook only renders `results/summary.json` and `results/metrics_per_case.csv`.

**Reading the numbers honestly:** the +/- figures are the standard deviation
**across patients**, not across random seeds. They describe how much the model's
performance varies case to case. They are *not* error bars on the mean and they
do not tell you how much the result would move if the model were retrained.

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import RESULTS_DIR, LOG_DIR

summary = json.loads((RESULTS_DIR / 'summary.json').read_text())
per_case = pd.read_csv(RESULTS_DIR / 'metrics_per_case.csv')
print('configs evaluated:', list(summary))
print('test patients   :', per_case["case_id"].nunique())

## Headline comparison

In [ ]:
rows = []
for name, s in summary.items():
    m = s['metrics']
    rows.append({
        'model': name,
        'params_M': round(s['parameters'] / 1e6, 2),
        'Dice WT': f"{m['WT']['dice_mean']:.3f} ± {m['WT']['dice_std']:.3f}",
        'Dice TC': f"{m['TC']['dice_mean']:.3f} ± {m['TC']['dice_std']:.3f}",
        'Dice ET': f"{m['ET']['dice_mean']:.3f} ± {m['ET']['dice_std']:.3f}",
        'HD95 WT': round(m['WT']['hd95_mean'], 1),
        'inference_s': round(s['inference_seconds_mean'], 2),
    })
display(pd.DataFrame(rows).set_index('model'))

## The ablation is the gap between these configurations

`baseline_unet` and `nnunet_style` share an architecture. They differ in three
things at once - normalisation, loss and augmentation - so the difference below
is the combined effect of that bundle, not an attribution to any one of them.
Separating them would need three more training runs.

In [ ]:
if {'baseline_unet', 'nnunet_style'} <= set(summary):
    b = summary['baseline_unet']['metrics']
    n = summary['nnunet_style']['metrics']
    for r in ['WT', 'TC', 'ET']:
        d = n[r]['dice_mean'] - b[r]['dice_mean']
        print(f'{r}: {b[r]["dice_mean"]:.3f} -> {n[r]["dice_mean"]:.3f}  ({d:+.3f})')
else:
    print('need both baseline_unet and nnunet_style results')

## Per-patient spread

The mean hides the interesting part. Enhancing tumour (ET) is bimodal: the model
either finds it or misses it almost entirely, which is why its standard deviation
is so much larger than whole tumour.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
configs = list(per_case['config'].unique())
for ax, region in zip(axes, ['WT', 'TC', 'ET']):
    ax.boxplot([per_case[per_case['config'] == c][f'{region}_dice'] for c in configs],
               labels=configs, showmeans=True)
    ax.set_title(f'{region} Dice'); ax.grid(alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=20, labelsize=8)
axes[0].set_ylabel('Dice')
fig.tight_layout(); plt.show()

## Training curves

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
for f in sorted(LOG_DIR.glob('*_history.json')):
    h = json.loads(f.read_text())
    a1.plot([p['iter'] for p in h['loss_curve']], [p['loss'] for p in h['loss_curve']],
            label=h['config'])
    a2.plot([p['iter'] for p in h['val_curve']], [p['mean'] for p in h['val_curve']],
            marker='o', label=h['config'])
a1.set(xlabel='iteration', ylabel='train loss', title='Training loss')
a2.set(xlabel='iteration', ylabel='mean Dice', title='Validation Dice (full volume)')
for ax in (a1, a2): ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Where it fails

The worst cases are the informative ones. Typical failure modes at this compute
budget: small enhancing regions missed entirely, and over-segmentation of oedema
into normal white matter hyperintensity, which looks similar on FLAIR.

In [ ]:
worst = (per_case.sort_values('WT_dice')
         .groupby('config').head(3)[['config','case_id','WT_dice','TC_dice','ET_dice']])
display(worst.round(3))

print('
Figures written by scripts/make_figures.py:')
for p in sorted((RESULTS_DIR / 'figures').glob('*.png')):
    print(' ', p.name)